# VRE model performance — TensorFlow 2.20 / AMD ROCm

This notebook is configured for the released paper-model reproduction: the complete 55-cell dataset, the fixed eight-cell test split, five selected features, and a five-member 70-epoch bootstrap ensemble. TensorFlow 2.20/ROCm can reproduce the method and target metrics, but it is not expected to be bitwise identical to the original TensorFlow 2.10 run.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import importlib
import os
import sys
import time

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name not in {'ML performance', 'ML-performance'}:
    for folder_name in ('ML-performance', 'ML performance'):
        candidate = NOTEBOOK_DIR / 'VRE_Trustability' / folder_name
        if candidate.is_dir():
            NOTEBOOK_DIR = candidate
            break
WORKSPACE_ROOT = NOTEBOOK_DIR.parents[1]
FEATURE_ROOT = WORKSPACE_ROOT / 'data' / 'processed' / 'features_linux_v1'
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
ARTIFACT_BASE = WORKSPACE_ROOT / 'artifacts' / 'rocm_tf220_paper_reproduction'
ARTIFACT_ROOT = ARTIFACT_BASE / RUN_TIMESTAMP
sys.path.insert(0, str(NOTEBOOK_DIR))

from functions import linux_model_pipeline as pipeline
importlib.reload(pipeline)
print('Features:', FEATURE_ROOT)
print('Run timestamp (UTC):', RUN_TIMESTAMP)
print('Artifacts:', ARTIFACT_ROOT)

Features: /home/phil30601/BatteryML/TrustworthyTest/data/processed/features_linux_v1
Run timestamp (UTC): 20261018T185114_203933Z
Artifacts: /home/phil30601/BatteryML/TrustworthyTest/artifacts/rocm_tf220_paper_reproduction/20261018T185114_203933Z


## 1. Runtime configuration and GPU preflight

Set `DEVICE_MODE` to `'gpu'`, `'cpu'`, or `'auto'`. GPU mode deliberately raises an error when ROCm is unavailable; CPU mode hides the GPU before TensorFlow initializes; auto mode uses a GPU when visible and otherwise falls back to CPU. After changing this setting, restart the Jupyter Python kernel and run from the first cell. The reproduction configuration below uses the released notebook values: five models, 70 epochs, batch size 256, SGD learning rate 0.001, momentum 0.001, and a 70% bootstrap sample per member.

In [2]:
# Change to 'cpu' to force CPU, 'gpu' to require ROCm GPU, or 'auto' to allow fallback.
DEVICE_MODE = 'cpu'

# Parameters copied from Model_Performance_3_10.ipynb cells 19, 20, and 22.
config = pipeline.SmokeConfig(
    seed=1,
    epochs=70,
    batch_size=128,
    bootstrap_fraction=0.70,
    num_models=5,
    learning_rate=0.001,
    momentum=0.001,
    steps_per_execution=1,  # reduce host scheduling overhead on the small dense model
    device_mode=DEVICE_MODE,
    deterministic_ops=False,  # original run did not enforce deterministic kernels
    max_train_rows=None,  # use every row in the training-cell split
)
tf = pipeline.configure_tensorflow(config)

TensorFlow: 2.20.0-dev0+selfbuilt
Requested device mode: cpu
Physical GPUs: []
Visible GPUs: []
Execution device: /CPU:0
Build: OrderedDict({'is_cuda_build': False, 'is_rocm_build': True, 'is_tensorrt_build': False})


2026-10-18 11:51:17.612644: E external/local_xla/xla/stream_executor/rocm/rocm_platform.cc:51] failed call to hipInit: HIP_ERROR_NoDevice


In [3]:
if config.device_mode == 'cpu':
    with tf.device('/CPU:0'):
        a = tf.random.normal((4096, 4096))
        start = time.perf_counter()
        b = tf.linalg.matmul(a, a)
        _ = b.numpy()
    print('CPU matmul seconds:', round(time.perf_counter() - start, 3))
    print('Tensor device:', b.device)
else:
    with tf.device('/GPU:0'):
        a = tf.random.normal((4096, 4096))
        start = time.perf_counter()
        b = tf.linalg.matmul(a, a)
        _ = b.numpy()
    print('GPU matmul seconds:', round(time.perf_counter() - start, 3))
    print('Tensor device:', b.device)

CPU matmul seconds: 0.157
Tensor device: /job:localhost/replica:0/task:0/device:CPU:0


## 2. Load only the five paper features

Paper mode requires all 55 feature partitions. It trains on the 47 non-test cells and evaluates only on cells `[3, 8, 15, 19, 30, 31, 34, 51]`. The five released-model features are `Energy`, `Capacity`, `Voltage-8`, `Energy-8`, and `Power-8`; arrays are converted to float32 immediately.

In [4]:
FULL_PAPER_SPLIT = True
SMOKE_TRAIN_CELLS = [1]
SMOKE_TEST_CELLS = [3]

loader = (
    pipeline.load_paper_split
    if FULL_PAPER_SPLIT
    else lambda root: pipeline.load_cell_split(
        root, SMOKE_TRAIN_CELLS, SMOKE_TEST_CELLS
    )
)
(
    x_train, y_train, x_test, y_test,
    train_index, test_index, train_cells,
) = loader(FEATURE_ROOT)
test_cells = pipeline.PAPER_TEST_CELLS if FULL_PAPER_SPLIT else SMOKE_TEST_CELLS
print('Features:', pipeline.PAPER_FEATURES)
print('Train cells:', train_cells)
print('Test cells:', test_cells)
print('Train:', x_train.shape, y_train.shape, x_train.dtype)
print('Test:', x_test.shape, y_test.shape, x_test.dtype)
print('Array memory GiB:', round((x_train.nbytes + y_train.nbytes + x_test.nbytes + y_test.nbytes) / 1024**3, 3))

Features: ['Energy', 'Capacity', 'Voltage-8', 'Energy-8', 'Power-8']
Train cells: [1, 2, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 55]
Test cells: [3, 8, 15, 19, 30, 31, 34, 51]
Train: (7693642, 5) (7693642,) float32
Test: (2162149, 5) (2162149,) float32
Array memory GiB: 0.22


## 3. Fit the scaler on training cells only

In [5]:
x_train, x_test, scaler = pipeline.scale_features(x_train, x_test, ARTIFACT_ROOT)
gc.collect()
print('Scaled train range:', float(x_train.min()), float(x_train.max()))

Scaled train range: -1.6325289011001587 1.9694772958755493


## 4. Train the five-member paper ensemble

Each member uses a point-wise bootstrap sample containing 70% of the training rows, sampled with replacement, and trains for 70 epochs at batch size 256. `steps_per_execution=64` groups 64 batches into each compiled execution to reduce host scheduling overhead without changing the optimizer update count. Every fresh execution of the initialization cell creates a UTC timestamped folder under `artifacts/rocm_tf220_paper_reproduction/`, preventing later runs from overwriting earlier models, scalers, bootstrap row IDs, predictions, or metrics.

In [6]:
start = time.perf_counter()
metrics, per_cell, y_pred = pipeline.train_smoke(
    tf=tf,
    x_train=x_train,
    y_train=y_train,
    x_test=x_test,
    y_test=y_test,
    test_index=test_index,
    artifact_dir=ARTIFACT_ROOT,
    config=config,
)
elapsed = time.perf_counter() - start
print('Elapsed seconds:', round(elapsed, 1))
print('RMSE: %.2f mV' % (metrics['rmse_v'] * 1000))
print('MAE: %.2f mV' % (metrics['mae_v'] * 1000))
print('Bias: %.2f mV' % (metrics['bias_v'] * 1000))
print('Pearson r: %.4f' % metrics['pearson_r'])
print('R2: %.4f' % metrics['r2'])
per_cell

Epoch 1/70
42075/42075 - 41s - 966us/step - loss: 0.0033 - mape: 549.4185 - mse: 0.0033
Epoch 2/70
42075/42075 - 38s - 909us/step - loss: 0.0016 - mape: 301.8367 - mse: 0.0016
Epoch 3/70
42075/42075 - 39s - 917us/step - loss: 0.0015 - mape: 287.5530 - mse: 0.0015
Epoch 4/70
42075/42075 - 38s - 909us/step - loss: 0.0014 - mape: 283.2537 - mse: 0.0014
Epoch 5/70
42075/42075 - 39s - 920us/step - loss: 0.0014 - mape: 273.6568 - mse: 0.0014
Epoch 6/70
42075/42075 - 39s - 920us/step - loss: 0.0013 - mape: 259.7942 - mse: 0.0013
Epoch 7/70
42075/42075 - 39s - 918us/step - loss: 0.0012 - mape: 245.8773 - mse: 0.0012
Epoch 8/70
42075/42075 - 38s - 914us/step - loss: 0.0011 - mape: 235.1017 - mse: 0.0011
Epoch 9/70
42075/42075 - 38s - 911us/step - loss: 0.0011 - mape: 229.1234 - mse: 0.0011
Epoch 10/70
42075/42075 - 38s - 911us/step - loss: 0.0011 - mape: 227.0618 - mse: 0.0011
Epoch 11/70
42075/42075 - 42s - 997us/step - loss: 0.0010 - mape: 226.9457 - mse: 0.0010
Epoch 12/70
42075/42075 - 38s 

{'3': {'rows': 814100,
  'rmse_v': 0.017465947922295617,
  'mae_v': 0.015632670372724533},
 '8': {'rows': 557242,
  'rmse_v': 0.015804154038442025,
  'mae_v': 0.011990098282694817},
 '15': {'rows': 94132,
  'rmse_v': 0.02785019323813907,
  'mae_v': 0.02030906267464161},
 '19': {'rows': 167087,
  'rmse_v': 0.035031817020936835,
  'mae_v': 0.031928651034832},
 '30': {'rows': 193011,
  'rmse_v': 0.027766570063424114,
  'mae_v': 0.02248385176062584},
 '31': {'rows': 76784,
  'rmse_v': 0.042423746177949306,
  'mae_v': 0.03423328325152397},
 '34': {'rows': 90403,
  'rmse_v': 0.03045413670943793,
  'mae_v': 0.01940133236348629},
 '51': {'rows': 169390,
  'rmse_v': 0.026388195287361898,
  'mae_v': 0.019064363092184067}}

## 5. Reproduction target and limitations

The stored output of the released notebook reports RMSE 0.02277 V, MAE 0.01710 V, and R² 0.84806 for the ensemble. Compare against these values only after the same feature preprocessing and all 55 cells are present. The released notebook seeds TensorFlow but does not seed NumPy before bootstrap sampling; this port fixes the seed so future runs are repeatable. TensorFlow/Keras and ROCm version differences can still produce small numerical differences.